# Zero-Shot Image Classification with CLIP

Companion notebook for the To Data & Beyond tutorial [Step-by-Step Guide to Zero-Shot Image Classification using CLIP Model](https://todatabeyond.com/blog/step-by-step-guide-to-zero-shot-image-classification-using-clip-model).

This notebook follows the complete article workflow with the official public `openai/clip-vit-large-patch14` checkpoint. It contains no credentials or saved execution outputs. The checkpoint is large, so a GPU runtime is recommended; CPU inference is possible but slower.

> CLIP is a research model. Review its model card, limitations, and potential biases before applying it to a real product or sensitive classification task.

## 1. Install dependencies

In [ ]:
%pip install -q transformers torch pillow requests

## 2. Load CLIP and its processor

The original archived notebook used a machine-local checkpoint path. This portable version uses the corresponding official Hugging Face model ID.

In [ ]:
import torch
from transformers import AutoProcessor, CLIPModel

model_id = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_id)
model.eval()
processor = AutoProcessor.from_pretrained(model_id)

## 3. Load and display the article image

The archived example image is loaded from its original Medium CDN URL so the notebook works without a local file.

In [ ]:
from io import BytesIO
import requests
from PIL import Image
from IPython.display import display

image_url = "https://cdn-images-1.medium.com/max/800/1*sipjo0osU7HhbObkUzioOA.png"
response = requests.get(image_url, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")
display(image)

## 4. Define candidate labels and prepare inputs

CLIP compares the image representation with text representations of labels supplied at inference time. The labels below preserve the concepts used in the original tutorial while correcting their grammar.

In [ ]:
labels = [
    "a photo of occupiers",
    "a photo of a cat",
]

inputs = processor(
    text=labels,
    images=image,
    return_tensors="pt",
    padding=True,
)

## 5. Run zero-shot classification

First inspect the complete model output if useful, then focus on the per-image logits and their softmax probabilities. Exact values can vary across library versions and hardware.

In [ ]:
with torch.inference_mode():
    outputs = model(**inputs)

outputs

In [ ]:
outputs.logits_per_image

In [ ]:
probs = outputs.logits_per_image.softmax(dim=1)[0]
probs

In [ ]:
for label, probability in zip(labels, probs.tolist()):
    print(f"label: {label} - probability: {probability:.4f}")

## Try your own labels

Replace `labels` with a fixed, meaningful taxonomy for your use case and rerun the input and inference cells. Prompt wording can affect scores, so test multiple appropriate descriptions and evaluate performance on representative data.